# TS2Vec Blood Glucose Prediction from PPG Signals

This notebook implements **two-stage training** for non-invasive glucose monitoring:

**Stage 1**: Self-supervised pre-training with TS2Vec contrastive learning  
**Stage 2**: Supervised fine-tuning with regression head on labeled glucose data

**Architecture**: TS2Vec (Dilated Convolutions)  
**Target Performance**: RMSE < 20 mg/dL, MAE < 15 mg/dL  
**Dataset**: VitalDB

Based on: "Non-invasive blood glucose monitoring using PPG signals" (2024)

## 0. Setup Environment

In [ ]:
# Clone repository with submodules (ts2vec is a submodule)
!cd /kaggle/working && rm -rf 3x1-PPG
!git clone --recurse-submodules -b dl-model --single-branch https://github.com/omar-A-hassan/3x1-PPG.git
!ls -la /kaggle/working/3x1-PPG/src/
!ls -la /kaggle/working/3x1-PPG/ts2vec/

print("Repository cloned successfully with ts2vec submodule")

In [ ]:
# Install dependencies from requirements.txt
!pip install -r /kaggle/working/3x1-PPG/requirements.txt -q

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Add project to path
sys.path.insert(0, '/kaggle/working/3x1-PPG')

# Import our modules
from src.models import TS2VecPPGTrainer, build_ts2vec_ppg, TS2VEC_AVAILABLE
from src.preprocessing import PPGPreprocessor
from src.training import Trainer
from src.evaluation import compute_all_metrics, print_metrics, clarke_error_grid_analysis

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"TS2Vec available: {TS2VEC_AVAILABLE}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Load VitalDB Dataset

Following the paper's methodology:
- 16-minute PPG windows (±8 min around glucose measurement)
- Downsampled to 100 Hz
- 1-second segments centered on systolic/diastolic peaks

In [ ]:
# Load VitalDB dataset
import vitaldb
from tqdm import tqdm

print("Loading VitalDB dataset with temporal alignment...")
print("16-minute PPG windows centered on glucose measurements\n")

# Load laboratory glucose data
df_labs = pd.read_csv('https://api.vitaldb.net/labs')
glucose_labs = df_labs[df_labs['name'] == 'gluc'].copy()
print(f"Found {len(glucose_labs)} glucose measurements")

# Find cases with both PPG and glucose
ppg_cases = set(vitaldb.find_cases(['SNUADC/PLETH']))
glucose_cases = set(glucose_labs['caseid'].unique())
common_cases = list(ppg_cases & glucose_cases)
print(f"Found {len(common_cases)} cases with both PPG and glucose\n")

# Limit for faster training (increase for full dataset)
MAX_CASES = 2000
if len(common_cases) > MAX_CASES:
    common_cases = np.random.choice(common_cases, MAX_CASES, replace=False).tolist()
    print(f"Using {MAX_CASES} cases for training")

In [ ]:
# Extract PPG-glucose pairs
ppg_signals = []
glucose_values = []
patient_ids = []
sampling_rate = 100  # Downsample to 100Hz

WINDOW_MINUTES = 16
WINDOW_SECONDS = WINDOW_MINUTES * 60
WINDOW_SAMPLES = WINDOW_SECONDS * sampling_rate  # 96,000 samples

for caseid in tqdm(common_cases, desc="Loading cases"):
    try:
        vals = vitaldb.load_case(caseid, ['SNUADC/PLETH'], 1/sampling_rate)
        if vals is None or len(vals) == 0:
            continue
        
        ppg_track = vals[:, 0]
        case_glucose = glucose_labs[glucose_labs['caseid'] == caseid]
        
        for _, glucose_row in case_glucose.iterrows():
            glucose_value = glucose_row['result']
            if glucose_value < 40 or glucose_value > 600:
                continue
            
            glucose_idx = int(glucose_row['dt'] * sampling_rate)
            half_window = WINDOW_SAMPLES // 2
            window_start = glucose_idx - half_window
            window_end = glucose_idx + half_window
            
            if window_start < 0 or window_end > len(ppg_track):
                continue
            
            ppg_window = ppg_track[window_start:window_end]
            if np.isnan(ppg_window).sum() / len(ppg_window) > 0.5:
                continue
            
            ppg_signals.append(ppg_window)
            glucose_values.append(glucose_value)
            patient_ids.append(caseid)
    except:
        continue

ppg_signals = np.array(ppg_signals)
glucose_values = np.array(glucose_values)
patient_ids = np.array(patient_ids)

print(f"\nDataset loaded: {len(ppg_signals)} samples from {len(np.unique(patient_ids))} patients")
print(f"Glucose range: {glucose_values.min():.1f} - {glucose_values.max():.1f} mg/dL")

## 2. Preprocess PPG Signals

Extract 1-second segments centered on peaks with template matching

In [ ]:
# Initialize preprocessor
preprocessor = PPGPreprocessor(
    sampling_rate=sampling_rate,
    segment_length=1.0,
    lowcut=0.5,
    highcut=8.0,
    peak_height_threshold=20,
    peak_distance_factor=0.8,
    similarity_threshold=0.85
)

print("Preprocessing PPG signals...")
print("Extracting 1-second peak-centered segments\n")

processed_segments = []
processed_glucose = []
processed_patient_ids = []

MAX_SEGMENTS_PER_MEASUREMENT = 200

for ppg_window, glucose, patient_id in tqdm(zip(ppg_signals, glucose_values, patient_ids), total=len(ppg_signals)):
    segments = preprocessor.preprocess(ppg_window, apply_template_matching=True)
    
    if segments is None or len(segments) == 0:
        continue
    
    # Subsample to match paper's approach
    if len(segments) > MAX_SEGMENTS_PER_MEASUREMENT:
        indices = np.random.choice(len(segments), MAX_SEGMENTS_PER_MEASUREMENT, replace=False)
        segments = [segments[i] for i in sorted(indices)]
    
    for segment in segments:
        processed_segments.append(segment)
        processed_glucose.append(glucose)
        processed_patient_ids.append(patient_id)

X = np.array(processed_segments, dtype=np.float32)
y = np.array(processed_glucose, dtype=np.float32)
patient_ids_processed = np.array(processed_patient_ids)

print(f"\nPreprocessed: {len(X)} 1-second segments")
print(f"Unique patients: {len(np.unique(patient_ids_processed))}")

## 3. Train/Val/Test Split by Patient

In [ ]:
from sklearn.model_selection import train_test_split

unique_patients = np.unique(patient_ids_processed)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)
train_patients, val_patients = train_test_split(train_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_ids_processed, train_patients)
val_mask = np.isin(patient_ids_processed, val_patients)
test_mask = np.isin(patient_ids_processed, test_patients)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train)} samples from {len(train_patients)} patients")
print(f"Val: {len(X_val)} samples from {len(val_patients)} patients")
print(f"Test: {len(X_test)} samples from {len(test_patients)} patients")

## 4. Stage 1: Pre-train TS2Vec Encoder (Self-Supervised)

Train with contrastive learning on unlabeled PPG data

In [ ]:
# Prepare unlabeled data for pre-training (3D: batch, seq_len, features)
X_pretrain = X_train[:, :, np.newaxis]  # (n_samples, 100, 1)

# Initialize trainer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ts2vec_trainer = TS2VecPPGTrainer(
    input_dims=1,
    output_dims=320,
    hidden_dims=64,
    depth=10,
    device=device
)

# Pre-train (Stage 1)
loss_log = ts2vec_trainer.pretrain(
    train_data=X_pretrain,
    n_iters=200,  # Increase for better pre-training
    batch_size=16,
    lr=0.001,
    verbose=True
)

# Save pre-trained encoder
ts2vec_trainer.save_pretrained('/kaggle/working/ts2vec_pretrained.pt')

## 5. Stage 2: Fine-tune with Regression Head (Supervised)

Freeze encoder and train regression head on labeled glucose data

In [ ]:
# Build regression model with pre-trained encoder
ts2vec_trainer.build_regression_model(freeze_encoder=True, dropout=0.1)

# Use our existing trainer for supervised fine-tuning
from src.training import train_model

trainer = train_model(
    ts2vec_trainer.model,
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size=32,
    epochs=100,
    learning_rate=0.001,
    device=device,
    save_dir='/kaggle/working/checkpoints'
)

## 6. Evaluate on Test Set

In [ ]:
# Load best model
trainer.load_checkpoint('/kaggle/working/checkpoints/best_model.pt')

# Predictions
ts2vec_trainer.model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred = ts2vec_trainer.model(X_test_tensor).cpu().numpy().squeeze()

# Compute metrics
metrics = compute_all_metrics(y_test, y_pred)
print_metrics(metrics)

## 7. Clarke Error Grid Analysis

In [ ]:
clarke_error_grid_analysis(
    y_test,
    y_pred,
    show_plot=True,
    save_path='/kaggle/working/clarke_error_grid.png'
)

## 8. Visualize Results

In [ ]:
# Training history
history = trainer.history

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_mae'], label='Train MAE')
axes[1].plot(history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (mg/dL)')
axes[1].set_title('MAE History')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=300)
plt.show()

print(f"Best validation MAE: {min(history['val_mae']):.2f} mg/dL")

In [ ]:
# Predictions plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5, s=30, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('True Glucose (mg/dL)')
plt.ylabel('Predicted Glucose (mg/dL)')
plt.title('Predicted vs True Glucose')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_pred - y_test
plt.scatter(y_test, residuals, alpha=0.5, s=30, edgecolors='k', linewidths=0.5)
plt.axhline(0, color='r', linestyle='--', linewidth=2)
plt.xlabel('True Glucose (mg/dL)')
plt.ylabel('Residual (mg/dL)')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/predictions.png', dpi=300)
plt.show()